In [4]:
# %pip install aieng-agents

In [ ]:
from aieng.agents.tools import CodeInterpreter
from pathlib import Path

# --- CONFIGURATION ---
DB_PATH = "/data"
AGENT_LLM_NAMES = {
    "worker": "gemini-2.5-flash",
    "manager": "gemini-2.5-pro",
}

# --- SHARED TOOLS ---
local_db = Path(DB_PATH)
shared_code_interpreter = CodeInterpreter(
    # template_name="lobsuu8phhb16r4r04yr", 
    template_name="q1sg157kmhnqbfjth0ue", 
    local_files=[local_db] if local_db.exists() else []
)

result = await shared_code_interpreter.run_code(
    code="""
import matplotlib.pyplot as plt
import numpy as np

x = np.linspace(0, 10, 100)
y = np.sin(x)

plt.plot(x, y)
plt.title("Sine Wave")
plt.savefig("sine_wave.png")
""",
)

# print(result.stdout)
# print(result.results)  # Contains base64 PNG data

/home/coder/eval-agents/.venv/lib/python3.12/site-packages/pydantic/main.py:528: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `str` - serialized value may not be as expected [field_name='results', input_value={'type': 'line', 'title':...'], 'y_scale': 'linear'}, input_type=dict])
  return self.__pydantic_serializer__.to_json(


In [2]:
print(result)  # Contains base64 PNG data

{"stdout":[],"stderr":[],"results":[{"text":"<PIL.Image.Image image mode=RGBA size=640x480>","png":"..."},{"text":"<Figure size 640x480 with 1 Axes>","png":"...","chart":{"type":"line","title":"Sine Wave","elements":[{"label":"Line 0","points":[[0.0,0.0],[0.10101010101010101,0.1008384202581046],[0.20202020202020202,0.2006488565226854],[0.30303030303030304,0.2984138044476411],[0.40404040404040403,0.3931366121483298],[0.5050505050505051,0.48385164043793466],[0.6060606060606061,0.5696341069089657],[0.7070707070707071,0.6496095135057065],[0.8080808080808081,0.7229625614794605],[0.9090909090909091,0.7889454628442574],[1.0101010101010102,0.8468855636029834],[1.1111111111111112,0.8961922010299563],[1.2121212121212122,0.9363627251042848],[1.3131313131313131,0.9669876227092996],[1.4141414141414141,0.9877546923600838],[1.5151515151515151,0.9984522269003895],[1.6161616161616161,0.9989711717233568],[1.7171717171717171,0.9893062365143401],[1.8181818181818181,0.9695559491823237],[1.9191919191919191,

In [3]:
type(result)

str

In [4]:
from aieng.agents import setup_langfuse_tracer, set_up_logging
from dotenv import load_dotenv

load_dotenv()
set_up_logging()

# Setup tracing
tracer = setup_langfuse_tracer(service_name="my_agent_app")

# Your agent code here - traces will automatically be sent to Langfuse

In [5]:
result = await shared_code_interpreter.run_code(
    code="""
import os
for root, dirs, files in os.walk('/data'):
    print(f"Path: {root}")
    print(f"Files: {files}")
"""
)
print(result)

{"stdout":["Path: /data","Files: ['cards_data.csv', 'mcc_codes.json', 'train_fraud_labels.json', 'transactions_data.csv', 'users_data.csv']"],"stderr":[],"results":null,"error":null}


In [6]:
initial_code = """
import pandas as pd
import json

# 1. Transactions: Safe because of nrows
try:
    tx = pd.read_csv('/data/transactions_data.csv', nrows = 50)
    print("Transactions Head:")
    print(tx.to_string())
except Exception as e:
    print(f"Error loading CSV: {e}")
"""

result = await shared_code_interpreter.run_code(code=initial_code)

In [7]:
# # 2. Labels: STREAM the file instead of loading it all
# print("\\nLabels Preview:")
# try:
#     with open('/data/train_fraud_labels.json', 'r') as f:
#         # We read just enough characters to see the start of the data
#         # rather than loading the whole multi-GB file.
#         preview = f.read(500) 
#         print(preview + "... [truncated]")
# except Exception as e:
#     print(f"Error loading JSON: {e}")

In [8]:
import json

result_str = await shared_code_interpreter.run_code(code=initial_code)

# Convert the string into a Python dictionary
result_json = json.loads(result_str)

# Now you can access keys like a normal object
if "error" in result_json and result_json["error"]:
    print(f"Error: {result_json['error']['name']}")
else:
    print("Success! Output:")
    print(result_json["stdout"])

Success! Output:
['Transactions Head:', '         id                 date  client_id  card_id   amount            use_chip  merchant_id    merchant_city merchant_state      zip   mcc  errors', '0   7475327  2010-01-01 00:01:00       1556     2972  $-77.00   Swipe Transaction        59935           Beulah             ND  58523.0  5499     NaN', '1   7475328  2010-01-01 00:02:00        561     4575   $14.57   Swipe Transaction        67570       Bettendorf             IA  52722.0  5311     NaN', '2   7475329  2010-01-01 00:02:00       1129      102   $80.00   Swipe Transaction        27092            Vista             CA  92084.0  4829     NaN', '3   7475331  2010-01-01 00:05:00        430     2860  $200.00   Swipe Transaction        27092      Crown Point             IN  46307.0  4829     NaN', '4   7475332  2010-01-01 00:06:00        848     3915   $46.41   Swipe Transaction        13051          Harwood             MD  20776.0  5813     NaN', '5   7475333  2010-01-01 00:07:00       18

In [9]:
type(result_json)

dict

In [1]:
# result_json['stdout']

In [11]:
type(result_json['stdout'])

list

In [12]:
len(result_json['stdout'])

52

In [2]:
# result_json['stdout'][9]

In [14]:
# #formatting the tranc csv in df

# import pandas as pd
# import re

# # 1. Identify the list from your previous result
# stdout = result_json['stdout']

# # 2. Define the correct columns manually to ensure nothing is missed
# columns = ['id', 'date', 'client_id', 'card_id', 'amount', 'use_chip', 
#            'merchant_id', 'merchant_city', 'merchant_state', 'zip', 'mcc', 'errors']

# # 3. Regular expression to capture each field accurately
# # This handles the index, the date space, and the 'Swipe Transaction' spaces
# pattern = r'^\d+\s+(\d+)\s+(\d{4}-\d{2}-\d{2}\s\d{2}:\d{2}:\d{2})\s+(\d+)\s+(\d+)\s+(\$[\-\d\.]+)\s+(.*?)\s{2,}(\d+)\s+(.*?)\s+([A-Z]{2})\s+([\d\.]+)\s+(\d+)\s+(.*)$'

# rows = []
# for line in stdout:
#     # We only process lines that start with a number (the data rows)
#     if line.strip() and line.strip()[0].isdigit() and '-' in line:
#         match = re.search(pattern, line)
#         if match:
#             rows.append(match.groups())

# # 4. Create the final DataFrame
# df_transactions = pd.DataFrame(rows, columns=columns)

# # 5. Optional: Clean numeric columns
# df_transactions['amount'] = df_transactions['amount'].str.replace('$', '').astype(float)
# df_transactions['id'] = df_transactions['id'].astype(int)

# # print("Transactions DataFrame (Full Columns):")
# # print(df_transactions[['id', 'merchant_id', 'merchant_city', 'merchant_state', 'amount']])

In [15]:
# df_transactions

### Agent for SQLite db file

In [16]:
import json
from typing import Any

import sqlglot
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


E2B_SQLITE_PATH = "/data/data.db"
MAX_SQL_ROWS = 50
_ALLOWED_ROOTS = {"select", "union", "paren"}
_FORBIDDEN_NODES = {
    "create",
    "insert",
    "update",
    "delete",
    "drop",
    "alter",
    "truncate_table",
    "merge",
    "command",
    "pragma",
    "attach",
    "detach",
    "set",
}


def _normalize_code_interpreter_result(raw_result: Any) -> dict[str, Any]:
    if isinstance(raw_result, str):
        return json.loads(raw_result)
    if isinstance(raw_result, dict):
        return raw_result
    if hasattr(raw_result, "model_dump"):
        return raw_result.model_dump()
    raise TypeError(f"Unsupported E2B result type: {type(raw_result)!r}")


def _extract_stdout_text(raw_result: Any) -> str:
    payload = _normalize_code_interpreter_result(raw_result)
    error = payload.get("error")
    if error:
        raise RuntimeError(f"E2B execution failed: {error}")

    stdout = payload.get("stdout", [])
    if isinstance(stdout, list):
        return "\n".join(str(line) for line in stdout)
    return str(stdout)


def _validate_read_only_sql(query: str) -> str:
    cleaned_query = query.strip().rstrip(";")
    if not cleaned_query:
        raise ValueError("SQL query must not be empty.")

    expressions = sqlglot.parse(cleaned_query, read="sqlite")
    if len(expressions) != 1:
        raise ValueError("Only a single SQL statement is allowed.")

    expression = expressions[0]
    if expression.key.lower() not in _ALLOWED_ROOTS:
        raise ValueError("Only read-only SELECT queries are allowed.")

    for node in expression.walk():
        if node.key.lower() in _FORBIDDEN_NODES:
            raise ValueError(f"Forbidden SQL operation detected: {node.key}")

    return cleaned_query


async def get_database_schema() -> str:
    """Return tables and columns available in the E2B SQLite database."""
    code = f"""
import sqlite3

conn = sqlite3.connect({E2B_SQLITE_PATH!r})
rows = conn.execute(
    \"\"\"
    SELECT name, type
    FROM sqlite_master
    WHERE type IN ('table', 'view')
      AND name NOT LIKE 'sqlite_%'
    ORDER BY type, name
    \"\"\"
).fetchall()

sections = []
for name, relation_type in rows:
    table_info = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
    columns = ', '.join(f"{{column[1]}} {{column[2] or 'TEXT'}}" for column in table_info)
    sections.append(f"{{relation_type}}: {{name}}\\n  columns: {{columns}}")

print('\\n\\n'.join(sections) or 'No tables found.')
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


async def run_sql_query(query: str) -> str:
    """Execute a read-only SQL query against the E2B SQLite database."""
    safe_query = _validate_read_only_sql(query)
    code = f"""
import json
import sqlite3

query = {safe_query!r}
conn = sqlite3.connect({E2B_SQLITE_PATH!r})
conn.row_factory = sqlite3.Row
cursor = conn.execute(query)
columns = [column[0] for column in (cursor.description or [])]
rows = [dict(row) for row in cursor.fetchmany({MAX_SQL_ROWS})]
payload = {{
    'columns': columns,
    'rows': rows,
    'row_limit': {MAX_SQL_ROWS},
    'query': query,
}}
print(json.dumps(payload, default=str))
conn.close()
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _extract_stdout_text(raw_result)


SQL_AGENT_INSTRUCTIONS = f"""
You are a lightweight SQL analyst for a SQLite database stored in an E2B sandbox at {E2B_SQLITE_PATH}.

Core behavior:
- When the user asks a data question in natural language, automatically inspect schema if needed, write SQL, execute it with run_sql_query, and return the answer.
- Only skip execution if the user explicitly asks for SQL only, query only, or no execution.
- Never invent table or column names.
- Only use read-only SQL.
- Keep answers concise and grounded in tool output.

Response format for normal data questions:
ANSWER:
- Give the direct answer first in plain English.

SQL:
```sql
<the exact SQL you executed>
```

RESULT:
- Summarize the returned rows clearly.
- Mention when results are capped at {MAX_SQL_ROWS} rows.

If execution fails:
- Explain the failure briefly.
- Fix the SQL and try again.
""".strip()

sql_agent = Agent(
    name="e2b_sql_agent",
    model=AGENT_LLM_NAMES["worker"],
    instruction=SQL_AGENT_INSTRUCTIONS,
    tools=[get_database_schema, run_sql_query],
)

sql_session_service = InMemorySessionService()
sql_runner = Runner(
    app_name=sql_agent.name,
    agent=sql_agent,
    session_service=sql_session_service,
)
sql_session = await sql_session_service.create_session(
    app_name=sql_agent.name,
    user_id="notebook-user",
    state={},
)


async def reset_sql_agent_session() -> str:
    """Start a fresh conversation with the SQL agent."""
    global sql_session
    sql_session = await sql_session_service.create_session(
        app_name=sql_agent.name,
        user_id="notebook-user",
        state={},
    )
    return sql_session.id


async def ask_sql_agent(question: str) -> str:
    """Ask the E2B-backed SQL agent a natural-language question and get the executed answer."""
    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks: list[str] = []

    async for event in sql_runner.run_async(
        user_id="notebook-user",
        session_id=sql_session.id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


async def answer_query(question: str) -> str:
    """Convenience wrapper for natural-language database questions."""
    return await ask_sql_agent(question)


print(f"SQL agent ready for {E2B_SQLITE_PATH}. Session: {sql_session.id}")

SQL agent ready for /data/data.db. Session: 5fed0acc-3246-4536-be85-951d2a569467


In [17]:
schema_preview = await get_database_schema()
print(schema_preview[:4000])

No tables found.


In [18]:
response = await answer_query(
    # "What are the top 5 merchant states by transaction count?"
    "Scan the database for high-frequency transaction fraud."
)
print(response)

/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/adk/runners.py:1481: DeprecationWarning: deprecated
  save_input_blobs_as_artifacts=run_config.save_input_blobs_as_artifacts,


21:04:28.116 invocation
21:04:28.119   invoke_agent e2b_sql_agent
21:04:28.125     call_llm


/home/coder/eval-agents/.venv/lib/python3.12/site-packages/google/genai/_api_client.py:755: DeprecationWarning: Inheritance class AiohttpClientSession from ClientSession is discouraged
  class AiohttpClientSession(aiohttp.ClientSession):  # type: ignore[misc]


21:04:29.224       execute_tool get_database_schema


21:04:30.527     call_llm
I cannot scan the database for high-frequency transaction fraud because there are no tables in the database.


In [20]:
query_preview = await run_sql_query(
    "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' ORDER BY type, name"
)
print(query_preview)

{"columns": ["name", "type"], "rows": [], "row_limit": 50, "query": "SELECT name, type FROM sqlite_master WHERE type IN ('table', 'view') AND name NOT LIKE 'sqlite_%' ORDER BY type, name"}


### Agent Dealing with RAW Files

In [21]:
# Template-aware agent for the new E2B file bundle.
# This cell does not modify the existing SQL agent; it adds a new one that can
# inspect all files currently available in the template and run a fraud-focused
# analysis over the raw files.

import json
from pathlib import PurePosixPath
from typing import Any

from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types


_TEMPLATE_ROOT = "/data"
_SUPPORTED_PREVIEW_SUFFIXES = {
    ".csv",
    ".json",
    ".jsonl",
    ".parquet",
    ".txt",
    ".md",
    ".sql",
    ".db",
    ".sqlite",
    ".sqlite3",
}


def _stdout_from_ci_result(raw_result: Any) -> str:
    if isinstance(raw_result, str):
        payload = json.loads(raw_result)
    elif isinstance(raw_result, dict):
        payload = raw_result
    elif hasattr(raw_result, "model_dump"):
        payload = raw_result.model_dump()
    else:
        raise TypeError(f"Unsupported CodeInterpreter result type: {type(raw_result)!r}")

    if payload.get("error"):
        raise RuntimeError(f"E2B execution failed: {payload['error']}")

    stdout = payload.get("stdout", [])
    if isinstance(stdout, list):
        return "\n".join(str(line) for line in stdout)
    return str(stdout)


async def list_template_files() -> str:
    """List files available in the current E2B template under /data."""
    code = f"""
import json
from pathlib import Path

root = Path({_TEMPLATE_ROOT!r})
if not root.exists():
    print(json.dumps({{"root": str(root), "files": [], "note": "root not found"}}))
else:
    files = []
    for path in sorted(root.rglob('*')):
        if path.is_file():
            files.append({{
                "path": str(path),
                "name": path.name,
                "suffix": path.suffix.lower(),
                "size_bytes": path.stat().st_size,
            }})
    print(json.dumps({{"root": str(root), "files": files}}, default=str))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _stdout_from_ci_result(raw_result)


async def preview_template_file(path: str, max_rows: int = 10) -> str:
    """Preview a supported file from the E2B template."""
    normalized_path = str(PurePosixPath(path))
    suffix = PurePosixPath(normalized_path).suffix.lower()
    if suffix not in _SUPPORTED_PREVIEW_SUFFIXES:
        raise ValueError(
            f"Unsupported file type for preview: {suffix}. Supported: {sorted(_SUPPORTED_PREVIEW_SUFFIXES)}"
        )

    code = f"""
import json
import sqlite3
from itertools import islice
from pathlib import Path

import pandas as pd

path = Path({normalized_path!r})
max_rows = int({max_rows})
suffix = path.suffix.lower()

if not path.exists():
    raise FileNotFoundError(f"File not found: {{path}}")

if suffix in {{'.csv'}}:
    df = pd.read_csv(path, nrows=max_rows)
    payload = {{"path": str(path), "kind": "table", "preview": df.to_dict(orient='records')}}
elif suffix in {{'.parquet'}}:
    df = pd.read_parquet(path).head(max_rows)
    payload = {{"path": str(path), "kind": "table", "preview": df.to_dict(orient='records')}}
elif suffix in {{'.json'}}:
    with path.open() as handle:
        data = json.load(handle)
    if isinstance(data, list):
        preview = data[:max_rows]
    elif isinstance(data, dict):
        preview = dict(list(data.items())[:max_rows])
    else:
        preview = data
    payload = {{"path": str(path), "kind": "json", "preview": preview}}
elif suffix in {{'.jsonl'}}:
    with path.open() as handle:
        preview = [json.loads(line) for line in islice(handle, max_rows)]
    payload = {{"path": str(path), "kind": "jsonl", "preview": preview}}
elif suffix in {{'.db', '.sqlite', '.sqlite3'}}:
    conn = sqlite3.connect(path)
    rows = conn.execute(
        \"\"\"
        SELECT name, type
        FROM sqlite_master
        WHERE type IN ('table', 'view')
          AND name NOT LIKE 'sqlite_%'
        ORDER BY type, name
        \"\"\"
    ).fetchall()
    preview = []
    for name, relation_type in rows:
        columns = conn.execute(f'PRAGMA table_info("{{name}}")').fetchall()
        preview.append({{
            "name": name,
            "type": relation_type,
            "columns": [{{"name": col[1], "type": col[2]}} for col in columns],
        }})
    conn.close()
    payload = {{"path": str(path), "kind": "sqlite", "preview": preview}}
else:
    text = path.read_text(errors='ignore')
    payload = {{"path": str(path), "kind": "text", "preview": text[:4000]}}

print(json.dumps(payload, default=str))
"""
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _stdout_from_ci_result(raw_result)


async def analyze_potential_fraud(limit: int = 20) -> str:
    """Rank potentially suspicious transactions from the raw template files."""
    safe_limit = max(1, min(int(limit), 100))
    code = """
import heapq
import json
from pathlib import Path

import pandas as pd

root = Path("__TEMPLATE_ROOT__")
transactions_path = root / 'transactions_data.csv'
cards_path = root / 'cards_data.csv'
users_path = root / 'users_data.csv'
mcc_path = root / 'mcc_codes.json'

cards = pd.read_csv(cards_path)
users = pd.read_csv(users_path)
with mcc_path.open() as handle:
    mcc_codes = json.load(handle)

cards = cards.rename(columns={'id': 'card_join_id'})
users = users.rename(columns={'id': 'user_join_id'})

transactions_usecols = [
    'id',
    'date',
    'client_id',
    'card_id',
    'amount',
    'use_chip',
    'merchant_id',
    'merchant_city',
    'merchant_state',
    'zip',
    'mcc',
    'errors',
]

amount_samples = []
chunk_size = 50000
max_rows_to_scan = 200000
rows_sampled = 0
for chunk in pd.read_csv(transactions_path, usecols=transactions_usecols, chunksize=chunk_size):
    rows_sampled += len(chunk)
    chunk['amount_numeric'] = (
        chunk['amount']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )
    amount_samples.extend(chunk['amount_numeric'].head(1000).tolist())
    if len(amount_samples) >= 50000 or rows_sampled >= max_rows_to_scan:
        break

if not amount_samples:
    raise RuntimeError('No transaction amounts were available for fraud analysis.')

sample_series = pd.Series(amount_samples, dtype='float64')
amount_threshold = float(sample_series.quantile(0.99))

high_risk_terms = [
    'money transfer',
    'wire transfer',
    'telecommunication',
    'travel',
    'miscellaneous',
    'digital',
    'online',
]

flagged_transactions = 0
total_transactions = 0
top_candidates = []
rows_processed = 0

def normalize_bool(series):
    return (
        series.fillna('')
        .astype(str)
        .str.strip()
        .str.lower()
        .isin(['yes', 'true', '1', 'y'])
    )

for chunk in pd.read_csv(transactions_path, usecols=transactions_usecols, chunksize=chunk_size):
    total_transactions += len(chunk)
    rows_processed += len(chunk)
    chunk['amount_numeric'] = (
        chunk['amount']
        .astype(str)
        .str.replace('$', '', regex=False)
        .str.replace(',', '', regex=False)
        .astype(float)
    )
    merged = chunk.merge(
        cards,
        left_on='card_id',
        right_on='card_join_id',
        how='left',
        suffixes=('', '_card'),
    )
    merged = merged.merge(
        users,
        left_on='client_id',
        right_on='user_join_id',
        how='left',
        suffixes=('', '_user'),
    )

    merged['mcc_description'] = merged['mcc'].astype(str).map(lambda value: mcc_codes.get(str(value), 'Unknown'))
    merged['errors_text'] = merged['errors'].fillna('').astype(str)
    merged['use_chip_text'] = merged['use_chip'].fillna('').astype(str)
    merged['merchant_state_text'] = merged['merchant_state'].fillna('').astype(str).str.upper()
    merged['card_on_dark_web_flag'] = normalize_bool(
        merged.get('card_on_dark_web', pd.Series(index=merged.index, dtype=object))
    )
    merged['high_amount_flag'] = merged['amount_numeric'] >= amount_threshold
    merged['error_flag'] = merged['errors_text'].str.strip().ne('')
    merged['online_flag'] = merged['use_chip_text'].str.contains('online', case=False, na=False)

    address_series = merged.get('address', pd.Series(index=merged.index, dtype=object)).fillna('').astype(str)
    state_match = address_series.str.extract(r',\\s*([A-Z]{2})\\s+\\d{5}(?:-\\d{4})?$', expand=False)
    merged['user_state_guess'] = state_match.fillna('').str.upper()
    merged['geo_mismatch_flag'] = (
        merged['user_state_guess'].ne('')
        & merged['merchant_state_text'].ne('')
        & merged['user_state_guess'].ne(merged['merchant_state_text'])
    )
    merged['high_risk_mcc_flag'] = merged['mcc_description'].str.lower().apply(
        lambda text: any(term in text for term in high_risk_terms)
    )

    merged['risk_score'] = (
        merged['card_on_dark_web_flag'].astype(int) * 5
        + merged['high_amount_flag'].astype(int) * 3
        + merged['geo_mismatch_flag'].astype(int) * 2
        + merged['error_flag'].astype(int) * 2
        + merged['online_flag'].astype(int) * 1
        + merged['high_risk_mcc_flag'].astype(int) * 1
    )

    flagged = merged[merged['risk_score'] > 0].copy()
    flagged_transactions += len(flagged)
    if flagged.empty:
        continue

    for _, row in flagged.iterrows():
        reasons = []
        if bool(row['card_on_dark_web_flag']):
            reasons.append('card_on_dark_web')
        if bool(row['high_amount_flag']):
            reasons.append('high_amount')
        if bool(row['geo_mismatch_flag']):
            reasons.append('state_mismatch')
        if bool(row['error_flag']):
            reasons.append('transaction_errors_present')
        if bool(row['online_flag']):
            reasons.append('online_transaction')
        if bool(row['high_risk_mcc_flag']):
            reasons.append('higher_risk_merchant_category')

        candidate = {
            'id': int(row['id']),
            'date': str(row['date']),
            'client_id': int(row['client_id']),
            'card_id': int(row['card_id']),
            'amount_numeric': float(row['amount_numeric']),
            'merchant_city': None if pd.isna(row['merchant_city']) else str(row['merchant_city']),
            'merchant_state': None if pd.isna(row['merchant_state']) else str(row['merchant_state']),
            'mcc': None if pd.isna(row['mcc']) else str(row['mcc']),
            'mcc_description': str(row['mcc_description']),
            'use_chip': None if pd.isna(row['use_chip']) else str(row['use_chip']),
            'errors': None if pd.isna(row['errors']) else str(row['errors']),
            'risk_score': int(row['risk_score']),
            'risk_reasons': reasons,
        }
        sort_key = (candidate['risk_score'], candidate['amount_numeric'], candidate['date'])
        if len(top_candidates) < __SAFE_LIMIT__:
            heapq.heappush(top_candidates, (sort_key, candidate))
        elif sort_key > top_candidates[0][0]:
            heapq.heapreplace(top_candidates, (sort_key, candidate))

    if rows_processed >= max_rows_to_scan:
        break

results = [item[1] for item in sorted(top_candidates, key=lambda item: item[0], reverse=True)]
payload = {
    'row_limit': __SAFE_LIMIT__,
    'total_transactions': int(total_transactions),
    'rows_scanned': int(rows_processed),
    'scan_limited': rows_processed >= max_rows_to_scan,
    'flagged_transactions': int(flagged_transactions),
    'high_amount_threshold': amount_threshold,
    'heuristics': {
        'card_on_dark_web': 'Card is marked as exposed or compromised.',
        'high_amount': 'Transaction amount is above the sampled 99th percentile threshold.',
        'state_mismatch': 'Merchant state differs from a state parsed from the user address.',
        'transaction_errors_present': 'Transaction has a non-empty errors field.',
        'online_transaction': 'Transaction channel is online.',
        'higher_risk_merchant_category': 'Merchant category description contains a higher-risk keyword.',
    },
    'results': results,
}
print(json.dumps(payload, default=str))
"""
    code = code.replace("__TEMPLATE_ROOT__", _TEMPLATE_ROOT)
    code = code.replace("__SAFE_LIMIT__", str(safe_limit))
    raw_result = await shared_code_interpreter.run_code(code=code)
    return _stdout_from_ci_result(raw_result)


TEMPLATE_AWARE_AGENT_INSTRUCTIONS = f"""
You are a template-aware analytics agent running against files available in the E2B sandbox.

Available tools:
- list_template_files: list all available files under {_TEMPLATE_ROOT}
- preview_template_file: inspect non-database files or inspect schemas for SQLite files
- analyze_potential_fraud: rank suspicious transactions from the raw fraud-analysis files

Behavior rules:
- Start with list_template_files when you are unsure which files exist.
- Use preview_template_file on newly available files before deciding how to answer.
- If the user asks for suspicious or potentially fraudulent transactions from the raw files, call analyze_potential_fraud.
- Prefer file-based reasoning and pandas-style analysis for raw CSV/JSON data.
- Never invent file names, columns, or tables.
- Return the answer directly for natural-language questions.
- Keep the answer concise and grounded in tool output.
- Include a short section naming which files you used.
""".strip()


template_aware_agent = Agent(
    name="template_aware_e2b_agent",
    model=AGENT_LLM_NAMES["worker"],
    instruction=TEMPLATE_AWARE_AGENT_INSTRUCTIONS,
    tools=[
        list_template_files,
        preview_template_file,
        analyze_potential_fraud,
    ],
)

template_aware_session_service = InMemorySessionService()
template_aware_runner = Runner(
    app_name=template_aware_agent.name,
    agent=template_aware_agent,
    session_service=template_aware_session_service,
)
template_aware_session = await template_aware_session_service.create_session(
    app_name=template_aware_agent.name,
    user_id="notebook-user",
    state={},
)


async def reset_template_aware_agent_session() -> str:
    """Reset the session for the template-aware agent."""
    global template_aware_session
    template_aware_session = await template_aware_session_service.create_session(
        app_name=template_aware_agent.name,
        user_id="notebook-user",
        state={},
    )
    return template_aware_session.id


async def answer_template_query(question: str) -> str:
    """Answer a natural-language question using the current E2B template files."""
    content = types.Content(role="user", parts=[types.Part(text=question)])
    final_chunks: list[str] = []

    async for event in template_aware_runner.run_async(
        user_id="notebook-user",
        session_id=template_aware_session.id,
        new_message=content,
    ):
        if not event.is_final_response() or not event.content:
            continue
        for part in event.content.parts:
            if part.text:
                final_chunks.append(part.text)

    return "\n".join(chunk for chunk in final_chunks if chunk).strip()


template_file_catalog = await list_template_files()
print(template_file_catalog)
print(f"Template-aware agent ready. Session: {template_aware_session.id}")

{"root": "/data", "files": [{"path": "/data/cards_data.csv", "name": "cards_data.csv", "suffix": ".csv", "size_bytes": 509860}, {"path": "/data/mcc_codes.json", "name": "mcc_codes.json", "suffix": ".json", "size_bytes": 4716}, {"path": "/data/train_fraud_labels.json", "name": "train_fraud_labels.json", "suffix": ".json", "size_bytes": 159083088}, {"path": "/data/transactions_data.csv", "name": "transactions_data.csv", "suffix": ".csv", "size_bytes": 1258531040}, {"path": "/data/users_data.csv", "name": "users_data.csv", "suffix": ".csv", "size_bytes": 164831}]}
Template-aware agent ready. Session: a5d2011f-59ed-4c28-93b8-08f0f12f7df7


In [22]:
example_template_response = await answer_template_query(
    "Using the new template files, summarize what data is available and explain how transactions_data.csv, cards_data.csv, users_data.csv, and mcc_codes.json could be combined for fraud analysis."
)
print(example_template_response)

21:06:06.920 invocation
21:06:06.922   invoke_agent template_aware_e2b_agent
21:06:06.924     call_llm
21:06:08.071       execute_tool list_template_files


21:06:09.487     call_llm
21:06:10.760       execute_tool preview_template_file


21:06:11.746     call_llm
21:06:12.777       execute_tool preview_template_file


21:06:14.621     call_llm
21:06:16.071       execute_tool preview_template_file


21:06:18.083     call_llm
21:06:19.156       execute_tool preview_template_file


21:06:20.824     call_llm
The available data includes `transactions_data.csv`, `cards_data.csv`, `users_data.csv`, and `mcc_codes.json`.

**Data Summary:**

*   **`transactions_data.csv`**: Contains individual transaction records, including `id`, `date`, `client_id`, `card_id`, `amount`, `use_chip`, `merchant_id`, `merchant_city`, `merchant_state`, `zip`, `mcc`, and `errors`.
*   **`cards_data.csv`**: Details card-specific information like `id` (card ID), `client_id`, `card_brand`, `card_type`, `card_number`, `expires`, `cvv`, `has_chip`, `num_cards_issued`, `credit_limit`, `acct_open_date`, `year_pin_last_changed`, and `card_on_dark_web`.
*   **`users_data.csv`**: Provides client (user) demographics and financial data, such as `id` (client ID), `current_age`, `retirement_age`, `birth_year`, `birth_month`, `gender`, `address`, `latitude`, `longitude`, `per_capita_income`, `yearly_income`, `total_debt`, `credit_score`, and `num_credit_cards`.
*   **`mcc_codes.json`**: A dictionary mappi

In [23]:
await reset_template_aware_agent_session()
example_template_response = await answer_template_query(
    "What files are available in /data, and which files would you use to join transactions to cardholders for fraud analysis?"
)
print(example_template_response)

21:06:27.075 invocation
21:06:27.076   invoke_agent template_aware_e2b_agent
21:06:27.077     call_llm
21:06:28.405       execute_tool list_template_files


21:06:29.984     call_llm
21:06:31.333       execute_tool preview_template_file


21:06:33.500     call_llm
21:06:34.519       execute_tool preview_template_file


21:06:36.287     call_llm
21:06:37.644       execute_tool preview_template_file


21:06:38.660     call_llm
The files available in `/data` are:
* `cards_data.csv`
* `mcc_codes.json`
* `train_fraud_labels.json`
* `transactions_data.csv`
* `users_data.csv`

To join transactions to cardholders for fraud analysis, I would use the following files:
*   `transactions_data.csv` (contains `client_id` and `card_id`)
*   `cards_data.csv` (contains `id` which can be joined with `transactions_data.csv` on `card_id`, and `client_id`)
*   `users_data.csv` (contains `id` which can be joined with `cards_data.csv` on `client_id`)


In [24]:
await reset_template_aware_agent_session()
example_template_response = await answer_template_query(
    "Find potentially fraudulent transactions using the available files. Prioritize cards marked on the dark web, unusually large purchases, and transactions that look geographically inconsistent with the user profile."
)
print(example_template_response)

21:06:41.518 invocation
21:06:41.519   invoke_agent template_aware_e2b_agent
21:06:41.520     call_llm
21:06:42.695       execute_tool analyze_potential_fraud


21:06:49.094     call_llm
Here are the top 20 potentially fraudulent transactions, ranked by risk score. The analysis prioritized cards on the dark web, unusually large purchases, and geographically inconsistent transactions.

**Heuristics Used for Fraud Detection:**
*   **card_on_dark_web**: Card is marked as exposed or compromised.
*   **high_amount**: Transaction amount is above the sampled 99th percentile threshold ($300.48).
*   **state_mismatch**: Merchant state differs from a state parsed from the user address.
*   **transaction_errors_present**: Transaction has a non-empty errors field.
*   **online_transaction**: Transaction channel is online.
*   **higher_risk_merchant_category**: Merchant category description contains a higher-risk keyword.

**Top Potentially Fraudulent Transactions:**

1.  **ID: 7612465**
    *   **Date:** 2010-02-05 01:20:00
    *   **Amount:** $356.88
    *   **Merchant:** ONLINE (Travel Agencies)
    *   **Errors:** Bad Card Number
    *   **Risk Score:*

In [25]:
await reset_template_aware_agent_session()
example_template_response = await answer_template_query(
    "Show the top 20 most suspicious transactions and include the risk factors used."
)
print(example_template_response)

21:06:58.719 invocation
21:06:58.720   invoke_agent template_aware_e2b_agent
21:06:58.722     call_llm
21:06:59.875       execute_tool analyze_potential_fraud


21:07:08.423     call_llm
Here are the top 20 most suspicious transactions:

| Transaction ID | Risk Score | Risk Reasons                                                                   |
|----------------|------------|--------------------------------------------------------------------------------|
| 7612465        | 7          | High amount, Transaction errors present, Online transaction, Higher risk merchant category |
| 7510376        | 7          | High amount, Transaction errors present, Online transaction, Higher risk merchant category |
| 7507721        | 7          | High amount, Transaction errors present, Online transaction, Higher risk merchant category |
| 7536966        | 6          | High amount, Transaction errors present, Online transaction                    |
| 7522752        | 6          | High amount, Transaction errors present, Online transaction                    |
| 7668064        | 6          | High amount, Transaction errors present, Higher risk merchant ca